# 06. Logogram Robustness

**Paper section:** §7 Logogram robustness (Table 6).
**What it computes:** Re-runs the segmentation pipeline on Akkadian / Sumerian texts after (a) leaving logograms intact and (b) explicitly masking logograms, to show that the script-level transitional-probability signal does not depend on logogram-rich passages. The Hittite version of the same check lives in notebook 05.
**Inputs:** Outputs of notebook 01.
**Outputs:** `outputs/table6_logogram_robustness.csv`.
**Expected runtime (CPU baseline):** ~5 min on CPU.

All randomness uses `SEED = 42`.

## Setup
This notebook uses shared utilities from `_setup.py`:
- `load_corpora()`: returns dict of dataframes per language
- `POS_HARMONIZATION`: dict mapping raw POS tags to unified tagset
- `BASE_PATH`: data directory (set this for your environment)

```python
from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH
```


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except ImportError:
    pass

from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH, set_seeds
set_seeds(42)


In [ ]:
# --- data-availability guard ------------------------------------------
missing = [l for l in ('akk', 'sux', 'elx') if l not in corpora]
if missing:
    print(f'WARNING: {missing} not loaded. Cells that depend on them will be skipped.')
    print(f'Set $CUNEI_DATA to a directory containing alltexts_AKK.csv, alltexts_SUX.csv, and the Elamite files.')
# Convenience: expose datasets dict for cells originally from the monolith.
datasets = {l: corpora[l] for l in ('akk', 'sux', 'elx') if l in corpora}
sign_dict = corpora.get('_sign_dict', {})


In [ ]:
corpora = load_corpora(BASE_PATH)
doc_corpora = corpora['_documents']


### Approach

Logograms are flagged in ORACC by uppercased / Sumerogram POS markers (`LN`, `MN`, `WN` plus all-caps lemmata). We build two masked variants of each document and rerun the segmenter.


In [ ]:
import re, json
from cunei_tools import CuneiSeg

def mask_logograms(doc, dataset, text_id):
    """Replace all-caps tokens (logograms) with a single placeholder."""
    rows = dataset[dataset['text_id'] == text_id]
    is_logo = rows['form_latin'].astype(str).str.match(r'^[A-Z\.\-]+$')
    masked = ' '.join('<LOGO>' if il else f for f, il in zip(rows['form_latin'], is_logo))
    return masked

results = {}
for lang in ('akk', 'sux'):
    if lang not in corpora:
        continue
    df = corpora[lang]
    docs_intact = doc_corpora[lang]['latin']
    docs_masked = {tid: mask_logograms(None, df, tid) for tid in docs_intact}
    seg = CuneiSeg(lang=lang)
    seg.train(list(docs_intact.values()))
    m_intact = seg.find_optimal_threshold(list(docs_intact.values()))
    seg2 = CuneiSeg(lang=lang)
    seg2.train(list(docs_masked.values()))
    m_masked = seg2.find_optimal_threshold(list(docs_masked.values()))
    results[lang] = {'intact': m_intact, 'masked': m_masked}

import os; os.makedirs('../outputs', exist_ok=True)
with open('../outputs/table6_logogram_robustness.csv', 'w') as f:
    f.write('language,condition,f1,precision,recall,threshold\n')
    for lang, conds in results.items():
        for cond_name, m in conds.items():
            f.write(f'{lang},{cond_name},{m["f1"]:.4f},{m["precision"]:.4f},{m["recall"]:.4f},{m["threshold"]:.2f}\n')
results


### Reference: Morfessor `corpusweight` robustness sweep

The other robustness check in the paper sweeps the Morfessor `corpusweight` hyperparameter to show the gap is not driven by hyperparameter tuning. That code lives in notebook 02; rerun it there if you want to reproduce Table 2's robustness column.
